# **Project 1: Global Baseline Predictors and RMSE**

Puja Roy

Summer 2026


**Business Perspective**

This project is about creating a recommender system that recommeneds Bollywood movies you will enjoy watching. You basically inform the recommender system how you liked the Bollywood movies you've already watched. Then, it predicts how you'll like the ones you haven't seen yet. The goal of the Bollywood movie recommender system is to solve how much you'll like a Bollywood movie you haven't seen.

This is done by applying two methods. One technique is to find the rating of a Bollywood movie. The other technique is to utilize a method that considers what you like and dislike about Bollywood movies. It also looks at whats good or bad about each Bollywood movie.

We check how good each method is by using Root Mean Squared Error. This helps us see how close the systems guesses are to your thoughts about a Bollywood movie. We use Root Mean Squared Error to find out which method is better, at guessing how much you'll like a Bollywood movie. The Root Mean Squared Error tells us which method is more accurate. This way we can see which one works better for suggesting Bollywood movies.

**Step 1: Create the Dataset**

In [1]:
# Import pandas
import pandas as pd

# Create Bollywood movie ratings dataset
ratings = pd.DataFrame({

    'UserID': [
        'U1','U1','U1','U1',
        'U2','U2','U2','U2',
        'U3','U3','U3','U3',
        'U4','U4','U4','U4',
        'U5','U5','U5','U5'
    ],

    'Movie': [
        '3 Idiots','Dangal','PK','Sholay',
        '3 Idiots','Dangal','Lagaan','Zindagi Na Milegi Dobara',
        'PK','Lagaan','Sholay','Zindagi Na Milegi Dobara',
        '3 Idiots','PK','Sholay','Lagaan',
        'Dangal','PK','Lagaan','Zindagi Na Milegi Dobara'
    ],

    'Rating': [
        5,5,4,3,
        4,5,3,4,
        5,4,2,5,
        3,2,4,5,
        4,3,4,5
    ]
})

print(ratings)

   UserID                     Movie  Rating
0      U1                  3 Idiots       5
1      U1                    Dangal       5
2      U1                        PK       4
3      U1                    Sholay       3
4      U2                  3 Idiots       4
5      U2                    Dangal       5
6      U2                    Lagaan       3
7      U2  Zindagi Na Milegi Dobara       4
8      U3                        PK       5
9      U3                    Lagaan       4
10     U3                    Sholay       2
11     U3  Zindagi Na Milegi Dobara       5
12     U4                  3 Idiots       3
13     U4                        PK       2
14     U4                    Sholay       4
15     U4                    Lagaan       5
16     U5                    Dangal       4
17     U5                        PK       3
18     U5                    Lagaan       4
19     U5  Zindagi Na Milegi Dobara       5


**Step 2: Create the User-Item Matrix**

In [2]:
# Create user-item matrix
user_item_matrix = ratings.pivot(
    index='UserID',
    columns='Movie',
    values='Rating'
)

print(user_item_matrix)

Movie   3 Idiots  Dangal  Lagaan   PK  Sholay  Zindagi Na Milegi Dobara
UserID                                                                 
U1           5.0     5.0     NaN  4.0     3.0                       NaN
U2           4.0     5.0     3.0  NaN     NaN                       4.0
U3           NaN     NaN     4.0  5.0     2.0                       5.0
U4           3.0     NaN     5.0  2.0     4.0                       NaN
U5           NaN     4.0     4.0  3.0     NaN                       5.0


The missing values indicate movies that have not yet been rated by a user.

**Step 3: Split Data into Training and Test Sets**

In [3]:
from sklearn.model_selection import train_test_split

# Split data
train_data, test_data = train_test_split(
    ratings,
    test_size=0.20,
    random_state=42
)

print("Training Observations:", len(train_data))
print("Testing Observations:", len(test_data))

Training Observations: 16
Testing Observations: 4


**Step 4: Calculate the Global Mean Rating**

In [4]:
# Calculate global average rating
global_mean = train_data['Rating'].mean()

print("Global Mean:", round(global_mean,4))

Global Mean: 3.8125


**Step 5: Raw Average Predictor**

The raw average predictor assigns the same value (the global mean) to every rating prediction.



In [12]:
# Predict every observation using global mean
train_data['Raw_Prediction'] = global_mean
test_data['Raw_Prediction'] = global_mean

**Step 6: Calculate RMSE for Raw Average Predictor**

In [6]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Training RMSE
train_rmse_raw = np.sqrt(
    mean_squared_error(
        train_data['Rating'],
        train_data['Raw_Prediction']
    )
)

# Testing RMSE
test_rmse_raw = np.sqrt(
    mean_squared_error(
        test_data['Rating'],
        test_data['Raw_Prediction']
    )
)

print("Training RMSE:", round(train_rmse_raw,4))
print("Testing RMSE:", round(test_rmse_raw,4))

Training RMSE: 0.9499
Testing RMSE: 1.1057


**Step 7: Calculate User Biases**

User bias measures how much a user's average rating differs from the global average.


In [7]:
# Calculate user means
user_means = train_data.groupby(
    'UserID'
)['Rating'].mean()

# Calculate user bias
user_bias = user_means - global_mean

print(user_bias)

UserID
U1   -0.312500
U2    0.187500
U3    0.187500
U4   -0.812500
U5    0.520833
Name: Rating, dtype: float64


**Step 8: Calculate Item Biases**

Item bias measures how much a movie's average rating differs from the global average.

In [8]:
# Calculate movie means
movie_means = train_data.groupby(
    'Movie'
)['Rating'].mean()

# Calculate item bias
item_bias = movie_means - global_mean

print(item_bias)

Movie
3 Idiots                   -0.312500
Dangal                      0.687500
Lagaan                     -0.145833
PK                         -0.145833
Sholay                     -0.812500
Zindagi Na Milegi Dobara    0.854167
Name: Rating, dtype: float64


**Step 9: Calculate Baseline Predictors**

The baseline prediction is:


```
r^ui​=μ+bu​+bi​
```

where:

*   μ = global average rating
*   bu = user bias
*   bi = movie bias

In [14]:
# Function to calculate baseline prediction
def baseline_predict(row):

    # User bias
    bu = user_bias.get(
        row['UserID'],
        0
    )

    # Movie bias
    bi = item_bias.get(
        row['Movie'],
        0
    )

    # Baseline prediction
    return global_mean + bu + bi


# Training predictions
train_data['Baseline_Prediction'] = train_data.apply(
    baseline_predict,
    axis=1
)

# Testing predictions
test_data['Baseline_Prediction'] = test_data.apply(
    baseline_predict,
    axis=1
)

**Step 10: Calculate RMSE for Baseline Predictor**

In [10]:
# Training RMSE
train_rmse_baseline = np.sqrt(
    mean_squared_error(
        train_data['Rating'],
        train_data['Baseline_Prediction']
    )
)

# Testing RMSE
test_rmse_baseline = np.sqrt(
    mean_squared_error(
        test_data['Rating'],
        test_data['Baseline_Prediction']
    )
)

print("Training RMSE:", round(train_rmse_baseline,4))
print("Testing RMSE:", round(test_rmse_baseline,4))

Training RMSE: 0.7969
Testing RMSE: 1.578


**Step 11: Compare the Models**

In [11]:
# Create comparison table
results = pd.DataFrame({

    'Model': [
        'Raw Average',
        'Baseline Predictor'
    ],

    'Training RMSE': [
        train_rmse_raw,
        train_rmse_baseline
    ],

    'Testing RMSE': [
        test_rmse_raw,
        test_rmse_baseline
    ]
})

print(results)

                Model  Training RMSE  Testing RMSE
0         Raw Average       0.949918      1.105738
1  Baseline Predictor       0.796858      1.577979


**Results**

The average predictor is really simple. It just looks at the rating from the training data for movies. The average predictor does not think about what individual users like or how popular a movie's. This is why the predictors predictions are not very good for movie ratings.

The baseline predictor is better than the predictor model. The baseline predictor takes into account what users like. What movies are popular with users. Some users like to give ratings and some users like to give low ratings to movies. The baseline predictor figures this out about users and movies. It also looks at movies to see if they are usually rated high or low by users.

We use something called RMSE to see how good each predictor model is at predicting movie ratings. We calculate the RMSE for both the predictor model and the baseline predictor model. The model with the RMSE is the better one because its predictions for movie ratings are closer to what users actually rated for movies. The average predictor and the baseline predictor are both used to predict movie ratings for users. The baseline predictor is an improvement over the predictor, for predicting movie ratings.

**Conclusion**

This project created a system that suggests Bollywood movies. It looked at how users rated movies. Two ways of doing this were tried: one way was to use a rating and another way was to think about how users and movies are different. The performance of both ways was checked by using something called RMSE on the groups of data used for training and testing. The results show that thinking about how users and moviesre different makes the suggestions more accurate. This is what people who know a lot about suggesting movies say should happen. When you think about how users and moviesre usually rated you get suggestions that are just right for each user and are accurate. The way that thinks about how users and moviesre different is a good starting point for making new models. These new models could use things like looking at the ratings closely and breaking down the big list of ratings, into smaller parts.